In [ ]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

grok_api_key = os.getenv("GROQ_API_KEY")
model_id = "llama-3.1-8b-instant"
groq_params = {
    "max_tokens": 256,
    "temperature": 0.5   ,
    "top_p": 0.9  
}

model = ChatGroq(
    model_name=model_id,
    api_key=grok_api_key,
    **groq_params
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize Components (All implement the Runnable interface)
prompt = ChatPromptTemplate.from_template("Tell me a brutal truth about {topic}.")
parser = StrOutputParser()

# 2. Compose the Chain (RunnableSequence)
# The pipe automatically handles the handoff from left to right.
chain = prompt | model | parser
response = chain.invoke({"topic": "training deep neural networks"})
print(response)

Here's a brutal truth about training deep neural networks:

**The majority of the training process is spent on overfitting, not underfitting.**

In other words, the biggest challenge in training deep neural networks is not making them too simple (underfitting), but rather making them too complex and fitting the noise in the training data (overfitting). This means that most of the time spent on training is wasted on optimizing the model to fit the training data perfectly, rather than generalizing well to new, unseen data.

This is because deep neural networks have a huge capacity to fit complex patterns in the data, but they also have a tendency to overfit and memorize the training data, rather than learning the underlying patterns and relationships. As a result, the model's performance on the test data is often poor, even if it performs well on the training data.

This is a brutal truth because it means that even with the most advanced hardware, the most sophisticated algorithms, and t

In [5]:
from langchain_core.runnables import RunnableParallel

# Assume we have two pre-defined chains
summary_chain = ChatPromptTemplate.from_template("Summarize: {text}") | model | parser
sentiment_chain = ChatPromptTemplate.from_template("What is the sentiment of: {text}?") | model | parser

# Type coercion automatically turns this dict into a RunnableParallel
parallel_chain = RunnableParallel({
    "summary": summary_chain,
    "sentiment": sentiment_chain
})

# Both LLM calls happen simultaneously.
# The output will be a dictionary with "summary" and "sentiment" keys.
results = parallel_chain.invoke({"text": "The loss curve is flatlining, and I'm out of compute credits."})
print(results)

{'summary': "It appears you're experiencing a plateau in model performance (flatlining loss curve) and are running out of computational resources (compute credits). This can be a challenging situation in machine learning, as it may indicate that the model has reached its optimal performance or that more data, better hyperparameters, or different architectures are needed to improve it.", 'sentiment': 'The sentiment of the given statement is one of frustration and possibly desperation. \n\n- The phrase "flatlining" implies a lack of progress or stagnation, which is a negative sentiment.\n- The phrase "I\'m out of compute credits" suggests that the person is unable to continue their work or project due to a lack of resources, which adds to the overall feeling of frustration. \n\nOverall, the sentiment is quite negative, indicating that the person is facing a difficult situation and is likely feeling stuck or unable to make progress.'}


In [6]:
# A standard Python function
def format_my_data(input_dict: dict) -> str:
    return input_dict['content'].upper()

# The function acts as a RunnableLambda within the chain
custom_chain = format_my_data | prompt | model | parser
response = custom_chain.invoke({"content": "neural networks"})
print(response)

Here's a brutal truth about neural networks:

**Neural networks are not as interpretable as you think, and their "black box" nature can be a significant limitation.**

While neural networks have achieved remarkable success in various applications, their internal workings are often opaque and difficult to understand. This lack of interpretability can make it challenging to:

1. **Identify the most important features**: Neural networks can be sensitive to a wide range of inputs, making it hard to pinpoint which features are most influential in their decision-making process.
2. **Understand the reasoning behind predictions**: When a neural network makes a prediction, it's often unclear why it arrived at that conclusion. This can lead to a lack of trust in the model's decisions.
3. **Debug errors and biases**: Without a clear understanding of how the network is processing inputs, it can be difficult to identify and address errors or biases in the model.
4. **Transfer knowledge to other dom

In [8]:
def get_word_count(input_dict: dict) -> int:
    return len(input_dict['content'].split())

def get_char_count(input_dict: dict) -> int:
    return len(input_dict['content'])

parallel_chain = RunnableParallel({
    "words": get_word_count,
    "chars": get_char_count,
    "upper": format_my_data
})

result = parallel_chain.invoke({"content": "neural networks"})
print(result)


{'words': 2, 'chars': 15, 'upper': 'NEURAL NETWORKS'}


In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 1. Construct the message history
messages = [
    SystemMessage(content="You are a brutal, highly concise AI/ML engineering mentor."),
    HumanMessage(content="I am struggling to understand backpropagation."),
    AIMessage(content="Backpropagation is just the chain rule of calculus applied to computational graphs. What specifically is confusing you?"),
    HumanMessage(content="How do the gradients update the weights?")
]

# 2. Invoke your Groq ChatModel directly
response = model.invoke(messages)
print(response.content)

**Weight Update**

Given:

- Loss function L
- Weights W
- Input X
- Output Y

**Backpropagation**

1. Forward pass: Calculate output Y using weights W and input X.
2. Calculate loss L using output Y.
3. Backward pass: Calculate gradients of loss with respect to weights (dL/dW).

**Weight Update Rule**

- New weights W_new = W_old - learning_rate * dL/dW

That's it. Focus on the math, not the implementation details.


In [10]:
# 1. Define the template using a list of tuples: (Role, Content)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}."),
    ("human", "Explain {concept} in one short sentence.")
])

# 2. Format the prompt (this creates the standard message list behind the scenes)
# Modern LangChain uses LCEL (the pipe operator |) to chain components.
chain = prompt | model | parser

# 3. Execute the chain
response = chain.invoke({"domain": "Deep Learning", "concept": "Transformer Attention mechanism"})
print(response)

The Transformer Attention mechanism is a neural network component that allows the model to focus on specific input elements (keys and values) relevant to the current element (query) being processed, weighing their importance through a weighted sum.


In [14]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# 1. Define your curated examples
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
]

# 2. Create a template for how a single example should look
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# 3. Build the Few-Shot template
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Inspect
print(few_shot_prompt.format())

Human: happy
AI: sad
Human: tall
AI: short


In [15]:
# 4. Assemble the final prompt, adding the system instructions and the actual user input
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a highly logical bot that provides antonyms."),
    few_shot_prompt,
    ("human", "{target_word}")
])

chain = final_prompt | model
response = chain.invoke({"target_word": "complex"})
print(response.content) # Expected output: "simple"

simple


In [16]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1. Initialize the parser
parser = CommaSeparatedListOutputParser()

# 2. Create the prompt, dynamically injecting the parser's instructions
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. \n{format_instructions}"),
    ("human", "List 5 essential Python libraries for {field}.")
])

# 3. Build the LCEL Chain: Prompt -> Model -> Parser
chain = prompt | model | parser
parser.get_format_instructions()

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [17]:
# 4. Invoke. Notice we pass the format_instructions from the parser into the prompt.
result = chain.invoke({
    "field": "Machine Learning", 
    "format_instructions": parser.get_format_instructions()
})

# The result is no longer a string, but a native Python list.
print(type(result)) # <class 'list'>
print(result)

<class 'list'>
['TensorFlow', 'Keras', 'Scikit-learn', 'PyTorch', 'OpenCV']


In [18]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

# 1. Define EXACTLY the structure we want using Pydantic
# The 'Field' descriptions are crucial—they tell the LLM how to populate each variable.
class PaperExtraction(BaseModel):
    title: str = Field(description="The main title of the research paper")
    authors: list[str] = Field(description="List of author names")
    core_algorithm: str = Field(description="The primary ML algorithm used (e.g., CNN, Transformer)")
    accuracy_score: float = Field(description="The reported accuracy as a decimal between 0 and 1")

# 2. Create the parser mapped to our custom schema
parser = PydanticOutputParser(pydantic_object=PaperExtraction)

# 3. Build the prompt. 
# We explicitly pass the parser's generated instructions into the System message.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an elite ML research assistant. Extract data from the user's text.\n\n{format_instructions}"),
    ("human", "{text}")
])

# 4. Construct the LCEL DAG (Directed Acyclic Graph)
# Data flows: Text Input -> Prompt -> Groq Llama 3 -> Pydantic Parser -> Typed Python Object
chain = prompt | model | parser

# 5. Invoke the chain
sample_text = """
We present 'Attention is All You Need', a novel architecture by Ashish Vaswani, Noam Shazeer, 
and Niki Parmar. We replace RNNs entirely with a Transformer architecture. On the WMT 2014 
English-to-German translation task, we achieved a BLEU score equivalent to 0.84 accuracy.
"""

In [19]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"description": "The main title of the research paper", "title": "Title", "type": "string"}, "authors": {"description": "List of author names", "items": {"type": "string"}, "title": "Authors", "type": "array"}, "core_algorithm": {"description": "The primary ML algorithm used (e.g., CNN, Transformer)", "title": "Core Algorithm", "type": "string"}, "accuracy_score": {"description": "The reported accuracy as a decimal between 0 and 1", "title": "Accuracy Score", "type": "number"}}, "required": ["title", "authors",

In [20]:
# Note: We must pass the format_instructions generated by the parser into the dictionary
result = chain.invoke({
    "text": sample_text,
    "format_instructions": parser.get_format_instructions()
})

# The result is a strictly typed Python object matching your exact specifications.
print(type(result))          # <class '__main__.PaperExtraction'>
print(result.title)          # Attention is All You Need
print(result.core_algorithm) # Transformer
print(result.accuracy_score) # 0.84

<class '__main__.PaperExtraction'>
Attention is All You Need
Transformer
0.84


# Sequential Chains (The Modern LCEL Way)

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Define the Prompts
prompt1 = ChatPromptTemplate.from_template("What is the most famous dish from {location}? Reply ONLY with the dish name.")
prompt2 = ChatPromptTemplate.from_template("Provide a brief, bulleted recipe for {meal}.")
prompt3 = ChatPromptTemplate.from_template("Based on this recipe, what is the estimated cooking time? \n{recipe}")

# 2. Define the Base Chains
# We parse the output to a string immediately so the next prompt can use it.
get_dish = prompt1 | model | StrOutputParser()
get_recipe = prompt2 | model | StrOutputParser()
get_time = prompt3 | model | StrOutputParser()

# 3. Construct the Sequential DAG using RunnablePassthrough.assign()
# This passes the dictionary along, adding new keys at each step.
sequential_chain = (
    # Step 1: Input is {"location": "China"}. We assign a new key "meal" by running get_dish.
    RunnablePassthrough.assign(meal=get_dish) 
    
    # Step 2: Input is now {"location": "...", "meal": "..."}. We assign "recipe".
    | RunnablePassthrough.assign(recipe=get_recipe)
    
    # Step 3: Input is now {"location", "meal", "recipe"}. We assign "time".
    | RunnablePassthrough.assign(time=get_time)
)

In [22]:
# 4. Invoke the master chain
results = sequential_chain.invoke({"location": "China"})

print(f"Dish: {results['meal']}")
print(f"Recipe: {results['recipe']}")
print(f"Time: {results['time']}")

Dish: Peking Duck.
Recipe: Here's a brief, bulleted recipe for Peking Duck:

**Ingredients:**

- 1 whole duck (3-4 lbs), patted dry
- 1/4 cup Chinese five-spice powder
- 2 tbsp sugar
- 2 tbsp soy sauce
- 2 tbsp Shaoxing wine (or dry sherry)
- 2 tbsp vegetable oil
- Scallions, sliced
- Pancakes (or steamed buns)
- Hoisin sauce
- Pickled scallions (optional)

**Instructions:**

- Preheat oven to 400°F (200°C).
- In a small bowl, mix five-spice powder, sugar, soy sauce, and Shaoxing wine.
- Rub the mixture all over the duck, then let it marinate for at least 2 hours or overnight.
- Roast the duck in the oven for 20 minutes, then increase heat to broil (high) for 5-7 minutes, or until the skin is crispy and golden brown.
- Let the duck rest for 10 minutes before slicing it into thin strips.
- Serve with pancakes, hoisin sauce, scallions, and pickled scallions (if using).

**Traditional Serving Method:**

-
Time: To estimate the cooking time, let's break down the recipe:

1. Marinating time

In [23]:
results

{'location': 'China',
 'meal': 'Peking Duck.',
 'recipe': "Here's a brief, bulleted recipe for Peking Duck:\n\n**Ingredients:**\n\n- 1 whole duck (3-4 lbs), patted dry\n- 1/4 cup Chinese five-spice powder\n- 2 tbsp sugar\n- 2 tbsp soy sauce\n- 2 tbsp Shaoxing wine (or dry sherry)\n- 2 tbsp vegetable oil\n- Scallions, sliced\n- Pancakes (or steamed buns)\n- Hoisin sauce\n- Pickled scallions (optional)\n\n**Instructions:**\n\n- Preheat oven to 400°F (200°C).\n- In a small bowl, mix five-spice powder, sugar, soy sauce, and Shaoxing wine.\n- Rub the mixture all over the duck, then let it marinate for at least 2 hours or overnight.\n- Roast the duck in the oven for 20 minutes, then increase heat to broil (high) for 5-7 minutes, or until the skin is crispy and golden brown.\n- Let the duck rest for 10 minutes before slicing it into thin strips.\n- Serve with pancakes, hoisin sauce, scallions, and pickled scallions (if using).\n\n**Traditional Serving Method:**\n\n-",
 'time': "To estimate th

# Memory (State Management)

In [24]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder

# 1. Create a prompt with a placeholder for history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="chat_history"), # History gets injected here
    ("human", "{user_input}")
])

chain = prompt | model | StrOutputParser()

# 2. Set up a store for session histories (In production, this is Redis or Postgres)
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 3. Wrap the chain with the history manager
conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="user_input",
    history_messages_key="chat_history",
)

In [25]:
# 4. Execute with a specific session ID
config = {"configurable": {"session_id": "session_xyz123"}}

res1 = conversational_chain.invoke({"user_input": "Hi, I'm an ML student from NUST."}, config=config)
print(res1)

Nice to meet you. NUST (National University of Sciences and Technology) is a reputable institution for pursuing a degree in Machine Learning (ML) and related fields. What specific area of ML are you interested in or currently working on?


In [26]:
res2 = conversational_chain.invoke({"user_input": "Where did I say I study?"}, config=config)
print(res2) # It will successfully recall "NUST"

You said you're an ML student from NUST, but you didn't specify the location of NUST. There are multiple institutions with the name NUST in different countries, such as Pakistan and Singapore. Could you please clarify which NUST you are referring to?


# Agents (Dynamic Execution)

In [36]:
import langchain

# 1. Turn on God Mode
langchain.debug = True

In [47]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# 1. Load some sample data
df = pd.DataFrame({
    "Region": ["North", "South", "East", "West"],
    "Sales": [50000, 60000, 45000, 70000],
    "Employees": [120, 150, 100, 180]
})

# 2. Create the Agent
# In modern LangChain, this agent lives in the 'experimental' package
# because it dynamically executes Python code.
agent_executor = create_pandas_dataframe_agent(
    llm=model,    
    df=df,              # The DataFrame you want it to analyze
    verbose=True,       # Crucial: Lets you see the LLM's thought process
    allow_dangerous_code=True, 
    agent_type="tool-calling", # Modern agent architecture
    return_intermediate_steps=True
)

In [48]:
# 3. Query the Agent
query = "What is the total number of employees in regions with sales greater than 55000?"

# The LLM will write `df[df['Sales'] > 55000]['Employees'].sum()` and execute it.
response = agent_executor.invoke({"input": query})

print(f"Final Answer: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df[df['Sales'] > 55000]['Employees'].sum()"}`


330The total number of employees in regions with sales greater than 55000 is 330.

> Finished chain.
Final Answer: The total number of employees in regions with sales greater than 55000 is 330.


In [49]:
df.loc[df['Sales'] > 55000, 'Employees'].sum()

np.int64(330)

In [50]:
response

{'input': 'What is the total number of employees in regions with sales greater than 55000?',
 'output': 'The total number of employees in regions with sales greater than 55000 is 330.',
 'intermediate_steps': [(ToolAgentAction(tool='python_repl_ast', tool_input={'query': "df[df['Sales'] > 55000]['Employees'].sum()"}, log='\nInvoking: `python_repl_ast` with `{\'query\': "df[df[\'Sales\'] > 55000][\'Employees\'].sum()"}`\n\n\n', message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'nb21w075j', 'function': {'arguments': '{"query":"df[df[\'Sales\'] \\u003e 55000][\'Employees\'].sum()"}', 'name': 'python_repl_ast'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq', 'finish_reason': 'tool_calls', 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand'}, id='lc_run--019d8b44-be5c-7d73-be30-fcebc20d326f', tool_calls=[{'name': 'python_repl_ast', 'args': {'query': "df[df['Sales'] > 55000]['

In [51]:
print("\n--- INTERMEDIATE STEPS (The Engine Room) ---")
for step in response['intermediate_steps']:
    action = step[0]  # The tool the LLM decided to call
    observation = step[1] # The result the tool returned
    
    print(f"Tool Called: {action.tool}")
    print(f"Code Executed: {action.tool_input}")
    print(f"Raw Output from Python: {observation}")


--- INTERMEDIATE STEPS (The Engine Room) ---
Tool Called: python_repl_ast
Code Executed: {'query': "df[df['Sales'] > 55000]['Employees'].sum()"}
Raw Output from Python: 330


In [55]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 2. Define your standard Python function
# This function takes a raw string, manipulates it, and returns a dictionary.
def preprocess_text(raw_input: str) -> dict:
    print(f"\n--- [Lambda Executing] Raw Input: '{raw_input}' ---")
    
    # Custom Logic: Strip whitespace, make uppercase, append context
    cleaned_topic = raw_input.strip().upper()
    augmented_topic = f"{cleaned_topic} in the context of Deep Learning"
    
    print(f"--- [Lambda Output] Formatted for Prompt: '{augmented_topic}' ---\n")
    
    # It MUST return a dictionary if the next step is a PromptTemplate
    return {"topic": augmented_topic}

# 3. Explicitly wrap the function in a RunnableLambda
# (Note: In modern LCEL, simply using the function name in the pipe automatically wraps it, 
# but making it explicit is best practice for clarity when starting out).
custom_transformer = RunnableLambda(preprocess_text)

# 4. Define the Prompt and Parser
prompt = ChatPromptTemplate.from_template("Explain {topic} in exactly one short sentence.")
parser = StrOutputParser()

# 5. Construct the LCEL Graph
# Data flows: String -> Python Function -> Prompt -> LLM -> String Output
chain = custom_transformer | prompt | model | parser

# 6. Invoke the chain with a messy input string
messy_user_input = "   backpropagation   "
final_answer = chain.invoke(messy_user_input)

print(f"Final LLM Output:\n{final_answer}")


--- [Lambda Executing] Raw Input: '   backpropagation   ' ---
--- [Lambda Output] Formatted for Prompt: 'BACKPROPAGATION in the context of Deep Learning' ---

Final LLM Output:
Backpropagation is a widely used algorithm in Deep Learning that efficiently computes the gradients of the loss function with respect to the model's parameters, enabling the optimization of these parameters through an iterative process.
